# Creating an evaluation function 

Enough theory, let's get to some code 😻

In this notebook, we shall examine an evaluation dataset (created by passing test data through the finetuned model & recording the predicted outputs) and create an evaluation function to generate metrics. 

## Imports 

- Import necessary libraries for handling JSON, structured data, and warnings.

- Load `calculate_metric` (a custom function for evaluating model performance) and `Function` and `Parameters` classes, which help structure function definitions. (see [utils/metric_types.py](./utils/metric_types.py) for explanations)

In [ ]:
import ast
import json
import pandas as pd

import warnings

from utils.metrics import calculate_metric
from utils.metric_types import Function, Parameters
warnings.simplefilter(action='ignore')

## Function Specifications 

### Load and Process Function Specifications

`raw_function_specs` is loaded from [../Data/function_spec.json](../Data/function_spec.json), which contains the expected function definitions. These definitions will serve as ground truth when evaluating the model’s output.

In [ ]:
# Load the function specifications - this file includes all the details for each function we have to conduct the evaluation on
with open('../Data/function_spec.json') as f:
    raw_function_specs = json.load(f)

Examine a single function from the loaded `raw_function_specs` list and  eye-ball it to check that function data is correctly loaded before evaluation, and to (re)aquaint yourself with the data. 

In [ ]:
# Get a sample function
sample_function = raw_function_specs[0]

sample_function

That might be a bit more verbose than you were expecting.  Try this:


In [ ]:
sample_function['function']

You might have noticed that the function spefications is in a particular format..

🤔 __Quick Question__: Why do you think that's the case?


<details>
    <summary>Answer: Click to show</summary>
    
The function spefication is in a standard json schema, which is often used for api calls, which has become the best practice.
</details>

### Restructure Function Specifications

Convert the `raw_function_specs` list into a dictionary (`function_specs`) - using the function name as the key, and the function details as values - so that we can look up function specifications by name.

In [ ]:
function_specs = {
    raw_function_spec['function']['name']: raw_function_spec['function']
    for raw_function_spec in raw_function_specs
}

Print the first three functions in a readable JSON format for verification.

In [ ]:
# See the processed function specs
for fn_name, fn_spec in list(function_specs.items())[:3]:
    print(f"---- Function: {fn_name}")
    print(json.dumps(fn_spec, indent=4))

## Load the Evaluation Dataset

A set of test prompts has been passed to the finetuned SLM, to create a dataset which contains: 
- The input prompt 
- The ground truth result (ie. expected function calls and parameters) for a given input. 
- The model’s generated output for the same input (ie. the prediction). 
- Additional functional metrics. 

Make yourself familiar with the data: 

In [ ]:
# Load prediction data
df = pd.read_csv("assets/prediction_with_metrics.csv")

# Fix formats
df['ground_truth'] = df['ground_truth'].apply(ast.literal_eval)
df['prediction'] = df['prediction'].apply(ast.literal_eval)

# Drop not useful columns
df = df.drop(columns=['id', 'explicitness', 'emotional_state'])

df.head()


## Detailed Functional Metrics

A reminder of the metrics we defined in the previous notebook 🤗

| Metric Name             | Definition          | 
|-------------------------|---------------------|
| passed	                | Did the model return the exact expected function? (1 = Yes, 0 = No) | 
| correct_function_called	| Did the model select the correct function?                     |
| function_exists	        | Does the function exist in the predefined specification?       |
| function_name_specified	| Did the model specify a function name?                         |
| correctly_generated_arguments	| Precision: Correct arguments / Total predicted arguments |
| correctly_retrieved_arguments	| Recall: Correct arguments / Total expected arguments     |
| extra_argument	              | Number of unexpected (hallucinated) arguments            |


# Define an Evaluation Function 

We will now define a function which calculates and returns evaluation metrics comparing the SLM-generated function call (ie. the output) to the ground truth function call. 

It uses `calculate_metric()` from [utils/metrics.py](./utils//metrics.py) to compute different metrics:
- Function name correctness – Does the model pick the correct function?
- Parameter correctness – Are the parameters correct in name and value?
- Other metadata – The function accepts timestamps (but they’re unused here).

#### Filtering Metrics:

If `functional_only=True`, it removes non-functional metrics like:
- secs_to_first_token (latency)
- secs_to_last_token (total time to generate)
- tokens_per_second (efficiency)

This ensures only accuracy-related function metrics are returned.

In [ ]:
def simple_metrics(ground_truth, model_answer, functional_only=True):
    metric_dict = calculate_metric(
        function_specs=function_specs,
        expected=Function(name=ground_truth['function_name'], parameters=Parameters(ground_truth['parameters'])),
        functions_from_slm=[Function(name=model_answer['function_name'], parameters=Parameters(model_answer['parameters']))],
        tokens_count=0,

        start_timestamp=None,
        first_token_timestamp=None,
        last_token_timestamp=None
    ).model_dump()

    if functional_only:
        non_functional_metrics = ['secs_to_first_token', 'secs_to_last_token', 'tokens_per_second']
        metric_dict = {key: value for key, value in metric_dict.items() if key not in non_functional_metrics}
        return metric_dict

    else:
        return metric_dict

## Example usage of the Evaluation Function 

Extract all function names from ground truth: 

In [ ]:
fn_names = []

for fn_name, _ in list(function_specs.items()):
    fn_names.append(fn_name)

fn_names 

And let's take one single sample from our dataset, and view the ground-truth vs. predicted result: 

In [ ]:

print('Ground Truth: ', df['ground_truth'][0])

print('Predicted: ', df['prediction'][0])


Now use our custom funtion to compare the predicted result with the ground truth and create the metrics.  

In [ ]:
simple_metrics(
    ground_truth=df['ground_truth'][0],
    model_answer=df['prediction'][0]
)

Let' breakdown the above results. 
- This output has `passed==1.0` and `correct_function_called==1.0`, meaning that the predicted function is indeed the same as the ground truth.

- `correctly_retrieved_arguments==1.0` the arguments have been retrieved correctly

- `function_name_specified==1.0`, meaning that the function name is specified in the predicted result, and it is also included in the list of `function_spec`

In [ ]:
df['ground_truth'][0].get('function_name') in fn_names

### Try it yourself 

Try it yourself now with another row of predictions: what do you observe?

In [ ]:
# Choose a random number < 400 and substitute n:
n = 0

simple_metrics(
    ground_truth=df['ground_truth'][n],
    model_answer=df['prediction'][n]
)

### Multiple samples
Let's repeat the same, but this time with multiple samples (or the entire dataset).  

In [ ]:
# Prepare sample data or full data
n = len(df)

sample_data = df[['ground_truth', 'prediction']].head(n)

Here we are considering the entire evaluation dataset. So, the predicted outputs from our finetuned model compared to the ground truth ones. We want to analyze how well did our model predict the given functions calls.

In [ ]:
sample_data

Let's calculate the metrics with the `simple_metric` function. As we already have the values of each of non functional metrics in the inference dataframe (`inference_metric.csv`), we will source them from there.

In [ ]:
results = sample_data.apply(lambda row: pd.Series(simple_metrics(row['ground_truth'], row['prediction'])), axis=1)
results.head()

In [ ]:
# Get the non functional metrics
results[['secs_to_first_token', 'secs_to_last_token', 'tokens_per_second']] = df[['time_to_first_token_seconds', 'total_time_seconds', 'tokens_per_second']].head(n)

In [ ]:
results.head()

In [ ]:
# Aggregated metrics for the sample data
print(f'Mean for {n} predictions, which is {(n/len(df)*100)}% of the entire eval dataset:')

pd.concat(
    [
        results.mean().rename('mean'),
    ],
    axis=1,
)

From the above we notice that our finetuned language model predicts quite well. Overall, more than 91% of the times, our model generated the correct function name and parameters.
- _mean_ corresponds to the average value for each metric in our predictions

In [ ]:
# Let's generate the final evaluation metrics results (both functional and non functional in one single dataframe)
final_results = pd.concat([sample_data, results], axis=1)
final_results.head()

Even though our model is quite good, it is meaningful to perform some _error_ analysis to understand where it makes mistakes and why these happen. Let's deep dive on it in the next section!

## Takeaways 

In this notebook we have learned:
- How evaluation is done for function calling 
- There are function and non functional metrics when evaluationg function calling, both are relevant and should be considered when measuring performance

Next Steps
- Identify common failure patterns.
- Improve function calling robustness by handling synonyms & partial correctness.
- Optimize non-functional performance by reducing latency.

### 🤔 __How would you conduct the Error Analysis based on these outputs?__ 

<details>
    <summary>Here are some ideas: Click to show</summary>

Go to the Error Analysis notebook called: [04-error-analysi.ipynb](04-error-analysis.ipynb)

</details>
